<a href="https://colab.research.google.com/github/yuzo-maruyama/quarto_stats/blob/main/finite_sum_JS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Theorem 1 for $3\leq p < p_*$

In [17]:
Pstar = 1000
Qstar = Pstar/2-1

In [ ]:
P = 3:(Pstar-1)
risk.diff = numeric(length(P))
for(k in 1:length(P)){
  my.ceiling =ceiling(sqrt(P[k]-2))
  risk = function(w){
        W=w/(P[k]-2)
        tmp.1 =1
        for(i in 1:my.ceiling){
          tmp.1=tmp.1+W^i
        }
        tmp.2 = w*(W^(my.ceiling)/tmp.1)^2
        tmp.2*dchisq(w,df=P[k])
  }
  tmp = integrate(risk, 0, Inf)
  risk.diff[k] = 2 - (tmp$value+tmp$abs.error)
}
sum(risk.diff>0); length(P)


[1] 1047

[1] 1047

# Theorem 1 for $p\geq p_*$

## region of the integrals

In [1]:
my.constant = 0.94*sqrt(2)

my.threshold = 6
c.1 = -log(2)/my.constant

## integrand

### integrand from $f$

$$k_1(x;q)=\frac{1}{\sqrt{2\pi}}(1+x/\sqrt{q})^q\exp(-\sqrt{q}x)$$

In [2]:
k.1 = function(x,q){
  (1/sqrt(2*pi))*exp(q*log1p(x/sqrt(q)) -sqrt(q)*x)
}

$$k_2(x;q)=\frac{1}{\sqrt{2\pi}}\exp(-x/\sqrt{q}-(1-1/q)x^2/2)$$

In [3]:
k.2 = function(x,q){
  (1/sqrt(2*pi))*exp(-x/sqrt(q)-(1-1/q)*x^2/2)
}

### $I_1$

$$
g_1(x)=x^2
 \frac{1-2\exp(c_1 x)}{(1-\exp(c_1 x))^2}
 $$

In [4]:
g.1 = function(x,mc){
  x^2*(1-2*exp(mc*x))/(1-exp(mc*x))^2
}

\begin{equation}
I_1(x;q) = \exp(-1/12q)g_1(x)*k_1(x;q)
\end{equation}

In [5]:
I.1 = function(x,q,mc){
  exp(-1/(12*q))*k.1(x,q)*g.1(x,mc)
}

### $I_2$

$$\mathrm{exprel.inv}(x)=\begin{cases} 1 & x=0 \\ \displaystyle\frac{x}{e^x-1}&x\neq 0\end{cases}$$

In [6]:
exprel.inv <- function(x) {
  ifelse(x == 0, 1, x/expm1(x))
}

$$
\begin{align}
g_2(x)&=x^2
\frac{2\exp(\sqrt{2} x)-1}{(\exp(\sqrt{2} x)-1)^2}\\
&=\frac{\exp(\sqrt{2}x)}{2}\left(\frac{\sqrt{2}x}{\exp(\sqrt{2}x)-1}\right)^2+\frac{x}{\sqrt{2}}\left(\frac{\sqrt{2}x}{\exp(\sqrt{2}x)-1}\right)\\
&=\frac{1}{2}\left(\exp(\sqrt{2}x)\mathrm{exprel.inv}(\sqrt{2}x)^2+(\sqrt{2}x)\mathrm{exprel.inv}(\sqrt{2}x)\right)\\
&=\frac{\mathrm{exprel.inv}(\sqrt{2}x)}{2}
\Bigl(\exp(\sqrt{2}x)\mathrm{exprel.inv}(\sqrt{2}x)+(\sqrt{2}x)\Bigr)\end{align}
$$

In [7]:
g.2=function(x,mc){
  y=mc*x
  exprel.inv(y)*(exprel.inv(y)*exp(y)+y)/mc^2
}

\begin{equation}
 I_2(x,q) =g_2(x)k_2(x;q)
%\frac{1}{\sqrt{2\pi}} x^2
%\frac{2\exp(\sqrt{2} x)-1}{(1-\exp(\sqrt{2} x))^2}
%\exp\left(-\frac{x}{\sqrt{q_*}} - \frac{q_*-1}{q_*}\frac{x^2}{2}%\right)
\end{equation}

In [8]:
I.2 = function(x,q,mc){
   g.2(x,mc)*k.2(x,q)
}

### $I_3$

$$\mathrm{powerrel.inv}(y;q)=\begin{cases} 1 & y=0 \\ \displaystyle\frac{y}{(1+y/\sqrt{2q})^{\sqrt{2q}}-1}&y\neq 0\end{cases}$$

In [9]:
powerrel.inv = function(y,q,mc){
  r=mc*sqrt(q)
  ifelse(y == 0, 1, y/expm1(r*log1p(y/r)))
}

$$
\begin{align}
g_3(x;q)&=x^2
\frac{2(1+x/\sqrt{q})^{\sqrt{2q}}-1}{((1+x/\sqrt{q})^{\sqrt{2q}}-1)^2}\\
&=\frac{(1+x/\sqrt{q})^{\sqrt{2q}}}{2}
\left(\frac{\sqrt{2}x}{((1+x/\sqrt{q})^{\sqrt{2q}}-1}\right)^2+\frac{x}{\sqrt{2}}
\left(\frac{\sqrt{2}x}{((1+x/\sqrt{q})^{\sqrt{2q}}-1}\right)\\
&=\frac{1}{2}\left((1+(\sqrt{2}x)/\sqrt{2q})^{\sqrt{2q}}\mathrm{powerrel.inv}(\sqrt{2}x)^2+(\sqrt{2}x)\mathrm{powerrel.inv}(\sqrt{2}x)\right)\\
&=\frac{\mathrm{powerrel.inv}(\sqrt{2}x)}{2}
\left((1+(\sqrt{2}x)/\sqrt{2q})^{\sqrt{2q}}\mathrm{powerrel.inv}(\sqrt{2}x)+(\sqrt{2}x)\right)
\end{align}
$$

In [10]:
g.3=function(x,q,mc){
  r=mc*sqrt(q)
  y=mc*x
  powerrel.inv(y,q,mc)*(exp(r*log1p(y/r))*powerrel.inv(y,q,mc)+y)/mc^2
}

\begin{equation}
 I_3(x,q)= g_3(x;q)*h_1(x,q)
% \frac{1}{\sqrt{2\pi}}
%x^2
%\frac{2(1+x/\sqrt{q_*})^{\sqrt{2q_*}}-1}{((1+x/\sqrt{q_*})^{\sqrt%{2q_*}}-1)^2}
%\exp(h(x;q_*))
\end{equation}

In [11]:
I.3 = function(x,q,mc){
  k.1(x,q)*g.3(x,q,mc)
}

## $I_4$

In [12]:
I.4 = function(q, mc){
  (2/mc^2)*exp(-(my.threshold-1/sqrt(q))^2/(1+my.threshold/sqrt(q))*(1/2))
}

## lower bound

In [40]:
Pstar = 10000000
Qstar = Pstar/2-1
my.constant = 0.94*sqrt(2)

my.threshold = 10
c.1 = -log(2)/my.constant

In [41]:
I.1.result <- integrate(I.1, -my.threshold, c.1, q=Qstar, mc=my.constant)
I.1.value <- I.1.result$value-I.1.result$abs.error

I.2.result <- integrate(I.2, c.1, 0, q=Qstar, mc=my.constant)
I.2.value <- I.2.result$value+I.2.result$abs.error

I.3.result <- integrate(I.3, 0, my.threshold, q=Qstar, mc=my.constant)
I.3.value <- I.3.result$value+I.3.result$abs.error

I.1.value - I.2.value- I.3.value -I.4(Qstar,my.constant)

[1] 0.0001654971

In [1]:
x=0.93926
1.3283/sqrt(2)

[1] 0.9392499